# Model Evaluation & Results Analysis

**Objective:** This notebook performs a comprehensive post-training evaluation of the PyTorch binary classification model. By analyzing the exported `training_history.csv` and `test_predictions.csv` from `train.py`, we assess model convergence, generalization capability, and diagnostic performance on unseen data.

In [ ]:
# =====================================================================
# Imports & Setup
# =====================================================================
# Import essential libraries for data manipulation (pandas, numpy) and 
# visualization (matplotlib, seaborn).
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import statistical metric functions from scikit-learn to quantitatively 
# evaluate the binary classification performance.
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_curve, 
    auc
)

# Set global plotting style for publication-ready visual consistency.
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

### 1. Data Ingestion
**Objective:** Load the artifacts generated during the neural network training phase. We require the epoch-by-epoch loss and F1 scores to check for overfitting, and the final test set probabilities to evaluate actual diagnostic power.

In [ ]:
# Load training history and test predictions into Pandas DataFrames.
# Using a try-except block ensures graceful error handling if the notebook 
# is executed in the wrong directory or before train.py finishes.
try:
    history_df = pd.read_csv('training_history.csv')
    preds_df = pd.read_csv('test_predictions.csv')
    
    # Confirming the dimensionality of the loaded data
    print("Data loaded successfully.")
    print(f"Loaded {len(history_df)} epochs of training history.")
    print(f"Loaded {len(preds_df)} test set predictions.")
except FileNotFoundError as e:
    print(f"Error loading files: {e}. Ensure the CSVs are in the same directory.")

### 2. Learning Curves Analysis
**Objective:** Visualize the model's learning dynamics. By plotting the Training vs. Validation Loss and F1 Scores across epochs, we can diagnose optimization issues. A diverging validation loss indicates **overfitting**, while a converging validation F1 score confirms that the model generalizes well to unseen data.

In [ ]:
# Create a 1x2 subplot grid to display Loss and F1 side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ---------------------------------------------------------
# Left Plot: Model Loss Over Epochs
# Tracks the optimization of the Cross-Entropy (or Label Smoothing) loss.
# ---------------------------------------------------------
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='Train Loss', color='blue', linewidth=2)
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='Val Loss', color='orange', linewidth=2)
axes[0].set_title('Model Loss Over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

# ---------------------------------------------------------
# Right Plot: Model F1 Score Over Epochs
# Tracks the harmonic mean of precision and recall. F1 is specifically 
# chosen here because it handles imbalanced medical datasets better than accuracy.
# ---------------------------------------------------------
axes[1].plot(history_df['epoch'], history_df['train_f1'], label='Train F1', color='blue', linewidth=2)
axes[1].plot(history_df['epoch'], history_df['val_f1'], label='Val F1', color='orange', linewidth=2)
axes[1].set_title('Model F1 Score Over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].legend()

# Adjust layout to prevent overlapping labels
plt.tight_layout()
plt.show()

### 3. Test Set Performance & Confusion Matrix
**Objective:** Quantify the model's hard predictions. The **Classification Report** provides granular metrics (Precision, Recall, F1) for both classes. The **Confusion Matrix** visually breaks down True Positives, True Negatives, False Positives (Type I errors), and False Negatives (Type II errors).

In [ ]:
# Extract the ground truth labels and the model's discrete predictions
y_true = preds_df['true_label']
y_pred = preds_df['predicted_label']

# Print standard machine learning classification metrics
print("=== Classification Report ===")
print(classification_report(y_true, y_pred, digits=4))

# Generate the Confusion Matrix to visualize exact error distributions
cm = confusion_matrix(y_true, y_pred)

# Plot the matrix using Seaborn's heatmap for readability
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 16})
plt.title('Test Set Confusion Matrix', fontsize=16)
plt.xlabel('Predicted Label', fontsize=14)
plt.ylabel('True Label', fontsize=14)
plt.show()

###  4. Receiver Operating Characteristic (ROC) & AUC
**Objective:** Evaluate the model's discriminative ability across *all* possible probability thresholds. The **AUC (Area Under the Curve)** represents the probability that the classifier will rank a randomly chosen positive instance higher than a randomly chosen negative one. An AUC closer to 1.0 indicates a highly separable, robust model.

In [ ]:
# Extract the raw, continuous probabilities assigned to the positive class (Class 1)
y_probs = preds_df['probability_class_1']

# Calculate the False Positive Rate (FPR) and True Positive Rate (TPR) at various probability thresholds
fpr, tpr, thresholds = roc_curve(y_true, y_probs)

# Compute the exact Area Under the Curve (AUC) score
roc_auc = auc(fpr, tpr)

# Plot the ROC Curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--') # Baseline random guess (AUC = 0.5)

# Set axis limits and labels
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

### 5. Error Analysis (Hard Examples)
**Objective:** Identify and isolate specific records where the model failed. By looking at high-confidence False Positives and low-confidence False Negatives, we can trace back to the raw source images to identify systematic biases, bad data, or complex clinical edge-cases that fooled the network.

In [ ]:
# Isolate Type I Errors (False Positives): Model predicted 1, but ground truth is 0.
false_positives = preds_df[(preds_df['true_label'] == 0) & (preds_df['predicted_label'] == 1)]

# Isolate Type II Errors (False Negatives): Model predicted 0, but ground truth is 1.
false_negatives = preds_df[(preds_df['true_label'] == 1) & (preds_df['predicted_label'] == 0)]

# Print aggregate error counts
print(f"Total False Positives: {len(false_positives)}")
print(f"Total False Negatives: {len(false_negatives)}\n")

# Display the 5 most "confident" False Positives (Model was very sure it was class 1, but was wrong)
print("--- Top 5 False Positives (High confidence, but actually class 0) ---")
display(false_positives.sort_values(by='probability_class_1', ascending=False).head(5))

# Display the 5 most "confident" False Negatives (Model gave a very low probability, but it was actually class 1)
print("\n--- Top 5 False Negatives (Low confidence, but actually class 1) ---")
display(false_negatives.sort_values(by='probability_class_1', ascending=True).head(5))